# ATLAS evaluation on a free Colab GPU

Runs the attack suites, firewall comparison, repeated-attack test and arena check with a local model. No API key is needed, and all tools and secrets are fake.

1. Choose **Runtime > Change runtime type > T4 GPU**, then Save.
2. Choose **Runtime > Run all**. If Colab warns the notebook isn't from Google, choose **Run anyway**.
3. Allow Google Drive access when asked. Results are saved to `ATLAS-reports` in your Drive as they finish, so a disconnect doesn't lose them.
4. Each cell prints a line per scenario, so you can watch progress. The last cell also downloads `atlas-reports.zip`.

In [ ]:
# Stops here if no GPU is attached, so nothing runs slowly by accident.
import shutil, subprocess
assert shutil.which('nvidia-smi') and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'No GPU. Choose Runtime > Change runtime type > T4 GPU, then Run all again.'
!nvidia-smi -L

In [ ]:
%cd /content
!rm -rf Atlas && git clone --quiet https://github.com/Itzdhruvpatel/Atlas.git
%cd /content/Atlas
!pip install --quiet -e '.[dev]'
!python -m pytest -q 2>&1 | tail -1

In [ ]:
# Saves reports to Google Drive as they finish. If Drive isn't allowed, they stay on this machine and the last cell still downloads them.
import os, shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
    target = '/content/drive/MyDrive/ATLAS-reports'
    os.makedirs(target, exist_ok=True)
    for name in os.listdir('eval/reports'):
        if name != '.gitkeep' and not os.path.exists(f'{target}/{name}'):
            shutil.copy(f'eval/reports/{name}', target)
    shutil.rmtree('eval/reports')
    os.symlink(target, 'eval/reports')
    print('Saving reports to Drive:', target)
except Exception as error:
    print('Drive not connected, saving on this machine only:', error)

# Installs Ollama, starts it and downloads the model (about 5 GB).
import time, urllib.request
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pkill ollama; sleep 2
# A bigger window, because this model can think for thousands of tokens before it acts.
subprocess.Popen('OLLAMA_CONTEXT_LENGTH=16384 OLLAMA_KEEP_ALIVE=-1 nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Read /content/ollama.log')
MODEL = 'lfm2.5:8b'
BASE = 'http://localhost:11434/v1'
!ollama pull {MODEL} 2>&1 | tail -1
!ollama run {MODEL} 'Say ready' 2>&1 | tail -1
!ollama ps

In [ ]:
# Stops the whole notebook here if replies get cut off or no attack lands, so no GPU time is wasted.
!python -m eval.diagnose --base-url {BASE} --model {MODEL} --ids A11,A13,A15 --trials 2
if _exit_code != 0:
    raise RuntimeError('Stopped to save GPU time. Read the STOP line above.')

In [ ]:
# No firewall. Prints one line per scenario.
!time python -m eval.baseline --suite strong --base-url {BASE} --model {MODEL} --runs 5 --max-llm-calls 400 --timeout 300
!time python -m eval.baseline --base-url {BASE} --model {MODEL} --runs 1 --max-llm-calls 200 --timeout 300

In [ ]:
# Firewall layers on and off. Saves after each setup.
!time python -m eval.ablation --suite strong --base-url {BASE} --model {MODEL} --runs 2 --max-llm-calls 2000 --timeout 300

In [ ]:
# Paper test: the same attack repeated in 1, 2 and 3 documents.
!time python -m eval.h1_spike --base-url {BASE} --model {MODEL} --trials 10 --max-llm-calls 1000 --timeout 300

In [ ]:
# Real rounds through the real arena API, one set per level.
!time python -m eval.arena_check --base-url {BASE} --model {MODEL} --trials 2

In [ ]:
# Downloads every saved report. Rerun this any time to get what has finished so far.
!cd /content/Atlas/eval/reports && zip -q -r /content/atlas-reports.zip . && ls -la
from google.colab import files
files.download('/content/atlas-reports.zip')